In [2]:
from rag.vectorstore import create_vectorstore
from rag.pdfloader import loaderpdf
import os

path = os.path.abspath("../documents/libro-del-curso.pdf")

chunk = loaderpdf(path)
create_vectorstore(chunk)

In [3]:
from rag.retriever import retrieve_context


query = "Explicame el objetivo principal del documento"

doc = retrieve_context(query)

print(doc)

c:\Users\tener\OneDrive\Escritorio\my_course_agent\src\rag\vectorstore.py:21: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


[Document(metadata={}, page_content='presenta una porción de datos es capaz de re cuperar todos los datos.  Para realizar una \nmemoria asociativa mediante una red recu rsiva (Hopfield propuso originalmente esta \naplicación para su red binaria), es necesario elegir los pesos de forma que produzcan un \nmínimo de energía en los vértices desea dos del hipercubo. Cada vector de estado \ncorrespondiente a un mínimo de energía se llama "memoria".  \nLa red parte de un estado inicial y el pr opio procedimiento de actualización mueve el'), Document(metadata={}, page_content='presenta una porción de datos es capaz de re cuperar todos los datos.  Para realizar una \nmemoria asociativa mediante una red recu rsiva (Hopfield propuso originalmente esta \naplicación para su red binaria), es necesario elegir los pesos de forma que produzcan un \nmínimo de energía en los vértices desea dos del hipercubo. Cada vector de estado \ncorrespondiente a un mínimo de energía se llama "memoria".  \nLa red part

In [ ]:
from langchain_ollama import OllamaLLM




llm = OllamaLLM(model="gemma:2b")


prompt = f"""
Usa este contexto:

{doc}

Pregunta: {query}
"""

llm.invoke(prompt)

In [ ]:


def map_step(query:str,chunks): #PRIMER PASO PARA MAP REDUCE
    summary_chunk = []

    for chunk in chunks:

        prompt = f"""
        Eres un asistente que analiza partes de un documento.

        Extrae SOLO la información relevante para responder esta pregunta:
        {query}

        Texto:
        {chunk}

        Respuesta breve:
        """

        rsp = llm.invoke(prompt)
        summary_chunk.append(rsp)
    return summary_chunk
    

def reduce_step(summary_chunk, query:str):# SEGUNDO PASO MAP REDUCE
    combined = "\n".join(summary_chunk)

    prompt = f"""
Eres un analista de documentos.

Con base en estos resúmenes parciales, responde la pregunta final de forma clara y directa.

Pregunta:
{query}

Resúmenes:
{combined}

Respuesta final:
"""
    return llm.invoke(prompt)


def map_reduce_rag(chunks, query): #PIPELINE YA INTEGRADO llamo ffunción en orden primero map_Step Para obtener vector debido que es aprametro necesario para la otra función, luego mando ese parametro al a funcion reduce y imprimo el retorno
    mapped = map_step(chunks, query)
    final_answer = reduce_step(mapped, query)
    return final_answer



response = map_reduce_rag(doc,query)

print(response)



